# Gold Layer Analytical Preview (Workshop 3)

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GOLD = ROOT / "datalake_gold"
FIGURES = ROOT / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

gov_files = sorted(GOLD.glob("governance_*.parquet"))
story_files = sorted(GOLD.glob("storytelling_*.parquet"))

latest_gov = gov_files[-1]
latest_story = story_files[-1]

df_gov = pd.read_parquet(latest_gov)
df_story = pd.read_parquet(latest_story)

print(f"Latest Governance file: {latest_gov.name} | Rows: {len(df_gov)} | Cols: {len(df_gov.columns)}")
print(f"Latest Storytelling file: {latest_story.name} | Rows: {len(df_story)} | Cols: {len(df_story.columns)}")

print("\n--- Governance KPIs Summary ---")
print(df_gov[["source", "dimension", "kpi_name", "kpi_value", "unit"]].to_string(index=False))

print("\n--- Storytelling Summary ---")
print(f"Unique metric_dates: {df_story['metric_date'].nunique()}")
print(f"Unique regions: {df_story['region_name'].nunique()}")
print(f"Total detections summed: {df_story['detection_count'].sum()}")
print(f"Total fire_event_count summed: {df_story['fire_event_count'].sum()}")
print(f"Total FRP (MW) summed: {df_story['total_frp_mw'].sum():.2f}")

print("\n--- Anomaly Level Breakdown ---")
print(df_story["anomaly_level"].value_counts(dropna=False))

print("\n--- Top 10 Region-Days by Severity ---")
cols_top = ["metric_date", "region_name", "detection_count", "fire_event_count", "total_frp_mw", "expected_detections", "anomaly_ratio", "anomaly_level", "daily_severity_rank"]
print(df_story.sort_values(["daily_severity_rank", "total_frp_mw"], ascending=[True, False])[cols_top].head(10).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

top_reg = (
    df_story.groupby("region_name", as_index=False)["total_frp_mw"]
    .sum()
    .sort_values("total_frp_mw", ascending=True)
    .tail(10)
)
axes[0, 0].barh(top_reg["region_name"], top_reg["total_frp_mw"], color="#d35400")
axes[0, 0].set_title("(a) Top 10 Departments by Cumulative FRP (MW)")
axes[0, 0].set_xlabel("Total FRP (MW)")

order_levels = ["Very low", "Low", "Normal", "Elevated", "Extreme"]
anom_counts = df_story["anomaly_level"].value_counts().reindex(order_levels, fill_value=0)
anom_counts.plot(kind="bar", ax=axes[0, 1], color=["#2980b9", "#27ae60", "#f1c40f", "#e67e22", "#c0392b"])
axes[0, 1].set_title("(b) Region-Day Anomaly Level Distribution")
axes[0, 1].set_ylabel("Region-Day Records")
axes[0, 1].tick_params(axis="x", rotation=20)

sc = axes[1, 0].scatter(
    df_story["temp_max_c"],
    df_story["detection_count"],
    c=df_story["rh_min_pct"],
    cmap="YlOrRd_r",
    alpha=0.8,
    edgecolors="k",
)
plt.colorbar(sc, ax=axes[1, 0], label="Min Relative Humidity (%)")
axes[1, 0].set_title("(c) Detections vs. Max Temperature & Min Humidity")
axes[1, 0].set_xlabel("Max Temperature (?C)")
axes[1, 0].set_ylabel("Detections")

sat_totals = pd.Series({
    "NOAA-20 (N20)": df_story["noaa20_count"].sum(),
    "NOAA-21 (N21)": df_story["noaa21_count"].sum(),
})
sat_totals.plot(kind="bar", ax=axes[1, 1], color=["#34495e", "#16a085"])
axes[1, 1].set_title("(d) Total Detections by VIIRS Satellite")
axes[1, 1].set_ylabel("Detections")
axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(FIGURES / "gold_storytelling_preview.png", dpi=150)
print(f"\nSaved figure to: {FIGURES / 'gold_storytelling_preview.png'}")


Latest Governance file: governance_20261122_052600.parquet | Rows: 38 | Cols: 7
Latest Storytelling file: storytelling_20261122_052600.parquet | Rows: 42 | Cols: 24

--- Governance KPIs Summary ---
         source            dimension                      kpi_name  kpi_value    unit
      firms_nrt                  N20        valid_ingestion_volume   138.0000 records
      firms_nrt                  N20          frp_outlier_rate_pct     2.1739 percent
      firms_nrt                  N20             revision_rate_pct     0.0000 percent
      firms_nrt                  N20    schema_compliance_rate_pct    97.8261 percent
      firms_nrt                  N20    median_ingestion_latency_h     9.7200   hours
      firms_nrt                  N21        valid_ingestion_volume   358.0000 records
      firms_nrt                  N21          frp_outlier_rate_pct     0.2793 percent
      firms_nrt                  N21             revision_rate_pct     0.0000 percent
      firms_nrt             


Saved figure to: C:\Proyectos\wildfire-monitoring\docs\figures\gold_storytelling_preview.png
